# Lab 4: Profiling Memory Access Patterns in LLMs

**Course**: *Dataflow Architectures for AI: From Principles to Practice*
**Week**: 4, Memory Systems
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)
**Runtime**: Google Colab, GPU runtime required (T4)
**Expected time**: 90–120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Capture per-operator allocation and deallocation with `torch.profiler` (`profile_memory=True`).
2. Identify the memory-intensive operators and the peak allocation during decode.
3. Measure *effective* memory bandwidth against an analytically known byte count, and explain why allocator activity is not bandwidth.
4. Derive the KV-cache size formula, confirm it against measured peak-memory growth, and size it for a scratchpad.
5. Predict where a scratchpad hierarchy (Spyre) helps versus a GPU cache, per bottleneck.

## Prerequisites
Labs 1-3 complete, plus the Week 4 lecture on memory hierarchies, the KV cache, and scratchpad vs cache.

## How this Self-Assessment Lab works

**Practice Lab**: You run this lab on **Google Colab** (free T4).  
**OPTIONAL SELF GRADING:** **A separate grader notebook is available on Coursera Labs for self-evaluation**.

| | Where | What happens |
|---|---|---|
| **1. This notebook** | Google Colab (T4 GPU) | Complete every `TODO`, make every **Checkpoint** pass, then run the final cell to write **`lab4_results.json`** |
| **2. Grader notebook** | Coursera Labs (no GPU) | For automated self assessment, paste the contents of `lab4_results.json`, then `Run All`. **40 ungraded points** |

The grader notebook has no GPU and no network, so it cannot re-run your
measurements. Instead it checks that your numbers are **internally consistent** —
the KV-cache byte count must follow from your model's own config, the effective
bandwidth from the bytes moved and the time taken, the per-token growth from the
peak-memory sweep you measured. Those relations hold for any real run on any
GPU, which is why a free-tier T4 earns a full score.

> **Students are encouraged to experiment with other GPUs and accelerators to expand their learning.** The free-tier T4 GPU is freely available on Google Colab. Re-running on an L4, V100 or A100 is genuinely instructive and you
> are encouraged to try it. There is **no extra credit** for doing so.

**Self-Assessment Lab ungraded points:**

| Part | Points | Comes from |
|---|---:|---|
| A — Structural invariants | 14 | Parts 3–7 of this notebook |
| B — Self-consistency | 18 | Parts 3–7 |
| C — Physics & plausibility | 8 | Parts 3–5 |
| **Ungraded notebook total** | **40** | fully auto-graded from your JSON for Self-Assessment |

**Graded Quiz points:**  
A 10 point graded Coursera Assignment quiz is given to assess your understanding from this lab experience.

| Quiz | Points | Comes from |
|---|---:|---|
| Concept Check quiz | 10 | Auto-graded Coursera Assignment |

The **Checkpoint** cells in this notebook are self-checks: they catch mistakes
as you work and assign no grade. Treat them as a preview — if a Checkpoint
fails here, the corresponding grader cell will fail there too.

### Working through this lab
1. Set **Runtime → Change runtime type → T4 GPU** before you run anything.
2. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
3. Fill in every `TODO`.
4. Run the final cell to produce **`lab4_results.json`**, then open the grader
   notebook on Coursera and paste it in for self-assessment.
5. Take the **Concept Check quiz** — a separate auto-graded Coursera
   Assignment worth **10 graded points**. It is mandatory, and you need 80% to
   pass.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                           ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## Part 0: Setup

We profile **`distilgpt2`** during a short generation so the **KV cache** grows
and shows up in the memory timeline — the Week 4 protagonist. The same harness
works on any causal LM.

> **Why distilgpt2?** It is small enough that a free T4 can sweep it to 512
> generated tokens in a couple of minutes, and its config is simple Multi-Head
> Attention (6 layers, 12 heads, head_dim 64), which makes the KV-cache formula
> you derive in Part 5b exact rather than approximate.

In [ ]:
!pip -q install transformers==4.44.2

import gc, math, json
import torch
import pandas as pd
from torch.profiler import profile, ProfilerActivity, record_function
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEV)
if DEV == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')

In [ ]:
MODEL_NAME = 'distilgpt2'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
# Load in FP16 on GPU. This is deliberate and the whole lab depends on it:
# every KV-cache figure below is quoted at 2 bytes per element, so the model has
# to actually BE half precision. Loading without torch_dtype gives FP32, and the
# measured per-token growth then comes out at twice the formula's prediction --
# a discrepancy that looks like a bug in the formula and is not.
_DTYPE = torch.float16 if DEV == 'cuda' else torch.float32
# `dtype=` is the current kwarg; `torch_dtype=` still works but warns on
# transformers >= 4.57. Fall back so the lab runs on either.
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=_DTYPE).to(DEV).eval()
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=_DTYPE).to(DEV).eval()
print('weight dtype:', next(model.parameters()).dtype)

PROMPT = 'Memory placement on a dataflow accelerator is decided by the compiler, because'
ids = tok(PROMPT, return_tensors='pt').input_ids.to(DEV)
print('prompt tokens:', ids.shape[-1])

---
## Part 1: Warm-up — a short generation

Generate a few tokens so kernels are compiled and the KV cache is populated
before profiling. A cold first call would charge kernel compilation to whichever
operator happened to run first.

In [ ]:
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=16, do_sample=False,
                         pad_token_id=tok.eos_token_id)
print('generated tokens:', out.shape[-1] - ids.shape[-1])
if DEV == 'cuda': torch.cuda.synchronize()

---
## Part 2: ★ TODO — Profile inference with memory tracking

Implement `profile_generation`: run a `max_new_tokens`-token generation inside a
`torch.profiler.profile` context with `profile_memory=True` and
`record_shapes=True`, recording the right activities for the device. Return the
`prof` object.

> Hint: `activities=[ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if cuda else [])`.

**💡 Hint: `profile_generation` (Part 2)**

<details>
<summary>Click to reveal a hint</summary>

```python
acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if ids.is_cuda else [])
with profile(activities=acts, profile_memory=True, record_shapes=True) as prof:
    with torch.no_grad():
        with record_function('generate'):
            model.generate(ids, max_new_tokens=new_tokens, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    if ids.is_cuda: torch.cuda.synchronize()
return prof
```

Two things that silently ruin the trace: leaving out `profile_memory=True`
(you still get a valid `prof`, just with every memory field zero), and
synchronizing *after* the `profile()` block exits instead of inside it.

</details>

In [ ]:
def profile_generation(model, ids, new_tokens=48):
    """Profile a short generation; return the profiler object."""
    # TODO 2.1: build the activity list for this device.
    # TODO 2.2: open a profile() context with profile_memory=True and
    #           record_shapes=True, run the generation inside it under
    #           torch.no_grad() and a record_function('generate') label,
    #           then synchronize if on CUDA.
    # TODO 2.3: return the profiler object.
    raise NotImplementedError("Implement profile_generation")

prof = profile_generation(model, ids, new_tokens=48)
print('profiling complete')

In [ ]:
# --- Checkpoint 1 -------------------------------------------------------
# The profiler must have captured events AND memory fields. A profile() without
# profile_memory=True still returns a valid object, so check the memory side.
assert prof is not None, "profile_generation returned None; did you return prof?"
_evts = list(prof.key_averages())
assert len(_evts) > 10, (
    f"only {len(_evts)} profiler events; a 48-token generation should produce "
    f"hundreds. Did the generate() call run inside the profile() context?")

def _mem_of(e):
    return (getattr(e, 'self_device_memory_usage', 0)
            or getattr(e, 'self_cuda_memory_usage', 0)
            or e.self_cpu_memory_usage or 0)

assert any(_mem_of(e) != 0 for e in _evts), (
    "no event reports memory usage. Pass profile_memory=True to profile().")
print(f"[OK] profiler captured {len(_evts)} distinct operators with memory fields")

---
## Part 3: ★ TODO — Peak allocation and memory-intensive operators

Implement `memory_table`: from `prof.key_averages()`, build a DataFrame of the
top operators by memory usage (use `self_device_memory_usage` on GPU, else
`self_cpu_memory_usage`), with their total time. Return the top `k`. Also record
**peak** allocation (`torch.cuda.max_memory_allocated` on GPU) as `peak_bytes`.

Your DataFrame must have exactly these columns, in this order:
`['op', 'self_mem_MB', 'time_ms', 'calls']`, sorted by `self_mem_MB` descending.

> **Why `getattr` with a fallback?** PyTorch renamed the CUDA-specific profiler
> fields to device-neutral ones (`self_cuda_memory_usage` →
> `self_device_memory_usage`) around 2.3. Reading both keeps the lab working on
> whatever torch Colab ships this month.

**💡 Hint: `memory_table` and `peak_bytes` (Part 3)**

<details>
<summary>Click to reveal a hint</summary>

```python
rows = []
for e in prof.key_averages():
    if cuda:
        mem = getattr(e, 'self_device_memory_usage', 0) or getattr(e, 'self_cuda_memory_usage', 0)
    else:
        mem = e.self_cpu_memory_usage
    t_us = getattr(e, 'device_time_total', 0) or getattr(e, 'cuda_time_total', 0) if cuda else e.cpu_time_total
    rows.append({'op': e.key, 'self_mem_MB': mem / 1e6,
                 'time_ms': t_us / 1e3, 'calls': e.count})
df = pd.DataFrame(rows)
df = df[df['self_mem_MB'].abs() > 0].sort_values('self_mem_MB', ascending=False)
return df.head(k).reset_index(drop=True)
```

For the peak, keep it in **raw bytes** — `int(torch.cuda.max_memory_allocated())`.
The grader notebook checks the units, and dividing by `1e6` here is the single
most common way to lose that point.

</details>

In [ ]:
def memory_table(prof, k=8, cuda=True):
    """Return a DataFrame of the top-k operators by memory usage."""
    # TODO 3.1: walk prof.key_averages(), reading the per-op self memory with a
    #           getattr fallback (device -> cuda -> cpu) and the matching time.
    # TODO 3.2: build a DataFrame with columns
    #           ['op', 'self_mem_MB', 'time_ms', 'calls'].
    # TODO 3.3: drop zero-memory rows, sort by self_mem_MB descending, and
    #           return the top k with a reset index.
    raise NotImplementedError("Implement memory_table")

cuda = (DEV == 'cuda')
top = memory_table(prof, k=8, cuda=cuda)

# TODO 3.4: peak allocation during the profiled generation, in RAW BYTES.
#           Hint: int(torch.cuda.max_memory_allocated()) on GPU, else 0.
peak_bytes = None  # REPLACE

if cuda:
    print(f'peak allocated: {peak_bytes/1e6:.1f} MB  ({peak_bytes:,} bytes)')
top

In [ ]:
# --- Checkpoint 2 -------------------------------------------------------
# Shape and units of the operator table, plus the peak figure the grader checks.
assert len(top) > 0, "memory_table returned no rows; did you filter out every op?"
assert list(top.columns) == ['op', 'self_mem_MB', 'time_ms', 'calls'], (
    f"expected columns ['op','self_mem_MB','time_ms','calls'], got {list(top.columns)}")
assert len(top) <= 8, f"you asked for k=8 but got {len(top)} rows"

# Sorted descending by memory: the top row is the dominant memory operator.
_mem = list(top['self_mem_MB'])
assert _mem == sorted(_mem, reverse=True), (
    "rows are not sorted by self_mem_MB descending; the grader reads top_ops[0] "
    "as your dominant memory operator.")

if cuda:
    # peak_bytes is raw BYTES, and must exceed the weights alone.
    _weight_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    assert peak_bytes is not None, "fill in peak_bytes (TODO 3.4)"
    assert peak_bytes > 1e6, (
        f"peak_bytes = {peak_bytes} looks like megabytes. Use the raw value "
        f"from torch.cuda.max_memory_allocated().")
    assert peak_bytes > _weight_bytes, (
        f"peak ({peak_bytes/1e6:.1f} MB) should exceed the weights alone "
        f"({_weight_bytes/1e6:.1f} MB): activations and the KV cache add to it.")
    print(f"[OK] {len(top)} operators ranked; peak {peak_bytes/1e6:.1f} MB "
          f"exceeds weights {_weight_bytes/1e6:.1f} MB")
else:
    print(f"[OK] {len(top)} operators ranked (CPU mode: peak_bytes not available)")
print(f"     dominant memory operator: {top.iloc[0]['op']}")

---
## Part 4: ★ TODO — Allocator activity vs. true memory bandwidth

The PyTorch profiler reports `self_device_memory_usage` (older torch:
`self_cuda_memory_usage`), which measures **net allocator activity**: bytes
allocated minus bytes freed, attributed to each operator.

**This is NOT memory bandwidth.** Net allocation can be zero for an in-place op
that streams gigabytes, negative for ops that free temporaries, and it does not
count reads at all. Dividing it by time gives "allocator activity per second",
not effective hardware bandwidth.

### Your task
1. Compute "allocator activity / time" from your profiler table — to see why it is unreliable. *(provided)*
2. Then run a **controlled benchmark** where the bytes transferred are analytically
   known: a `torch.copy_` of a known-size tensor. A copy of *N* bytes reads *N* and
   writes *N*, so the traffic is exactly **2N**. *(your code)*
3. Compare the two numbers. They will differ by roughly an order of magnitude, and
   the measured bandwidth will land **below** your GPU's spec peak — both of which
   are the point.

### Measurement Best Practices

> **⚠️ Single-run timings can be misleading.** GPU performance varies due to thermal throttling,
> background processes, and memory state. Always use multiple measurements.

**Protocol for reliable benchmarks:**
1. **Warmup:** 3-5 iterations (discarded) to fill caches and trigger JIT compilation
2. **Repeat:** N ≥ 10 measurement iterations
3. **Report:** Median (robust to outliers) ± IQR or percentiles (P5, P50, P95)
4. **Synchronize:** Always call `torch.cuda.synchronize()` before timing GPU ops
5. **Isolate:** Use `torch.inference_mode()` or `torch.no_grad()` to avoid autograd overhead

In [ ]:
# Utility: robust timing with statistics
#
# PROVIDED in the student notebook too (identical code) - not graded, not
# exported. Same helper as Lab 1; Lab 6 builds on it again.

import time
import numpy as np

def bench_robust(fn, n_warmup=5, n_measure=20, sync_cuda=True):
    """Run fn() with warmup, return timing statistics in milliseconds.

    Why percentiles and not just a mean: GPU timings are noisy and the noise is
    one-sided (thermal throttling and contention only ever make a run slower),
    so the distribution has a long right tail. A mean chases that tail; the
    median does not. The P5-P95 spread and the IQR tell you whether a number is
    stable enough to quote at all.
    """
    # Warm-up: fill caches, trigger lazy init and any JIT compilation, then
    # throw the results away. None of that is what we want to measure.
    for _ in range(n_warmup):
        fn()
    if sync_cuda and torch.cuda.is_available():
        torch.cuda.synchronize()

    # Measure: synchronize on both sides of each timed region, because CUDA
    # kernel launches are asynchronous and would otherwise be timed as ~0.
    times = []
    for _ in range(n_measure):
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    return {
        "median_ms": float(np.median(times)),
        "mean_ms":   float(np.mean(times)),
        "std_ms":    float(np.std(times)),
        "p5_ms":     float(np.percentile(times, 5)),
        "p95_ms":    float(np.percentile(times, 95)),
        "iqr_ms":    float(np.percentile(times, 75) - np.percentile(times, 25)),
        "n":         n_measure,
    }

# Demo + self-test on a plain matmul.
if torch.cuda.is_available():
    x = torch.randn(1024, 1024, device=DEVICE)
    stats = bench_robust(lambda: torch.mm(x, x))
    print(f"Matrix multiply (1024x1024) on {DEVICE}:")
    print(f"  Median: {stats['median_ms']:.3f} ms")
    print(f"  P5-P95: [{stats['p5_ms']:.3f}, {stats['p95_ms']:.3f}] ms")
    print(f"  IQR:    {stats['iqr_ms']:.3f} ms (over {stats['n']} runs)")
    assert "median_ms" in stats and stats["n"] == 20
    assert stats["p5_ms"] <= stats["median_ms"] <= stats["p95_ms"]
    print("  [OK] bench_robust returns sane statistics")
else:
    print("Skipping bench_robust demo (no GPU)")


In [ ]:
# 4.1: Allocator activity per time (NOT true bandwidth).  [PROVIDED]
# Kept deliberately: seeing the wrong number next to the right one is the lesson.
bw_proxy = top.copy()
bw_proxy['alloc_activity_GBps'] = bw_proxy.apply(
    lambda r: abs(r['self_mem_MB']) * 1e6 / 1e9 / (r['time_ms'] / 1e3)
    if r['time_ms'] > 0 else float('nan'), axis=1)
print("Allocator activity / time (NOT effective bandwidth):")
print(bw_proxy[['op', 'self_mem_MB', 'time_ms', 'alloc_activity_GBps']]
      .head(5).to_string(index=False))

# The proxy for the single dominant operator, exported so the grader notebook can
# confirm it is nowhere near the measured bandwidth.
_timed = bw_proxy[bw_proxy['time_ms'] > 0]
alloc_proxy_GBps = float(_timed.iloc[0]['alloc_activity_GBps']) if len(_timed) else float('nan')
print(f"\n⚠  These are NOT hardware bandwidth; they measure net allocation, not traffic.")
print(f"   Top timed operator's proxy: {alloc_proxy_GBps:.2f} GB/s")

**💡 Hint: `measure_copy_bandwidth` (Part 4.2)**

<details>
<summary>Click to reveal a hint</summary>

```python
nbytes = int(size_mb * 1e6)
src = torch.randn(nbytes // 4, device=device)   # float32 = 4 bytes
dst = torch.empty_like(src)
for _ in range(3):                              # warm up
    dst.copy_(src)
if device == 'cuda': torch.cuda.synchronize()

times = []
for _ in range(n_trials):
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    dst.copy_(src)
    if device == 'cuda': torch.cuda.synchronize()
    times.append(time.perf_counter() - t0)

median_s    = sorted(times)[len(times) // 2]
bytes_moved = 2 * nbytes            # <-- read src AND write dst
bw_gbs      = (bytes_moved / 1e9) / median_s
return bw_gbs, median_s * 1e3, bytes_moved
```

**The factor of 2 is the whole point of the exercise.** A copy reads the source
and writes the destination, so a 64 MB copy moves 128 MB. Leave it out and you
report exactly half the real bandwidth — Checkpoint 3 and the grader notebook
both check for it.

</details>

In [ ]:
# 4.2: ★ TODO - controlled bandwidth benchmark with analytically known traffic.
def measure_copy_bandwidth(size_mb=64, n_trials=20, device='cpu'):
    """Measure effective bandwidth using a controlled tensor copy.

    Known traffic: the copy reads src (size_mb) and writes dst (size_mb), so
    total traffic = 2 * size_mb. That factor of 2 is the whole reason this
    number means something and the Part 4.1 proxy does not.

    Returns (bw_gbs, median_ms, bytes_moved).
    """
    # TODO 4.2a: allocate a src tensor of size_mb and a dst of the same shape.
    #            float32 is 4 bytes, so nbytes // 4 elements.
    # TODO 4.2b: warm up with a few untimed copies, then synchronize.
    # TODO 4.2c: time n_trials copies with torch.cuda.synchronize() on BOTH
    #            sides of each timed region, and take the median.
    # TODO 4.2d: bytes_moved = 2 * nbytes  (read + write - do not drop the 2).
    #            bw_gbs = (bytes_moved / 1e9) / median_seconds
    # TODO 4.2e: return (bw_gbs, median_ms, bytes_moved).
    raise NotImplementedError("Implement measure_copy_bandwidth")

COPY_SIZE_MB = 64
copy_bw_gbs, copy_time_ms, copy_bytes_moved = measure_copy_bandwidth(
    size_mb=COPY_SIZE_MB, device=DEV)

print(f"Controlled copy benchmark ({DEV}, {COPY_SIZE_MB} MB):")
print(f"  Bytes moved (analytic): {copy_bytes_moved:,} = 2 x {COPY_SIZE_MB} MB")
print(f"  Median time:            {copy_time_ms:.3f} ms")
print(f"  Effective bandwidth:    {copy_bw_gbs:.1f} GB/s")
print()
print(f"  Allocator proxy (4.1):  {alloc_proxy_GBps:.2f} GB/s   <- not bandwidth")
print(f"  Measured (4.2):         {copy_bw_gbs:.1f} GB/s   <- IS bandwidth")
print()
_T4_SPEC_GBPS = 320.0        # NVIDIA T4 datasheet, GDDR6 peak
if cuda and 'T4' in torch.cuda.get_device_name(0):
    print(f"  A T4's spec peak is {_T4_SPEC_GBPS:.0f} GB/s, so you reached "
          f"{copy_bw_gbs/_T4_SPEC_GBPS*100:.0f}% of it.")
print("  A real copy never reaches the datasheet number: DRAM efficiency,")
print("  refresh and request overhead are not free. This is why Week 6's")
print("  roofline uses a MEASURED ceiling, not a quoted one.")

In [ ]:
# --- Checkpoint 3 -------------------------------------------------------
# The bandwidth number must be self-consistent and physically sane.
assert copy_bytes_moved == 2 * int(COPY_SIZE_MB * 1e6), (
    f"bytes_moved = {copy_bytes_moved:,} but a {COPY_SIZE_MB} MB copy moves "
    f"2 x {COPY_SIZE_MB} MB = {2*int(COPY_SIZE_MB*1e6):,} bytes (read + write). "
    f"Forgetting the factor of 2 halves your reported bandwidth.")
assert copy_time_ms > 0, "median copy time must be positive"

# Recompute bandwidth from the two raw inputs; they must agree.
_bw_check = (copy_bytes_moved / 1e9) / (copy_time_ms / 1e3)
assert abs(_bw_check - copy_bw_gbs) / _bw_check < 0.02, (
    f"bw_gbs = {copy_bw_gbs:.1f} does not follow from bytes_moved / time "
    f"(= {_bw_check:.1f}). The grader notebook recomputes this.")

if cuda:
    # Wide band: anything from a throttled T4 to an A100.
    assert 10 < copy_bw_gbs < 3000, (
        f"{copy_bw_gbs:.1f} GB/s is outside the plausible range for a GPU copy.")
    # The lesson check: the proxy and the real measurement must not coincide.
    assert alloc_proxy_GBps < copy_bw_gbs, (
        f"the allocator proxy ({alloc_proxy_GBps:.2f} GB/s) came out at or above "
        f"the measured bandwidth ({copy_bw_gbs:.1f} GB/s). Recheck 4.1: the proxy "
        f"is net allocation per unit time, not traffic.")
print(f"[OK] bandwidth self-consistent: {copy_bw_gbs:.1f} GB/s from "
      f"{copy_bytes_moved:,} bytes in {copy_time_ms:.3f} ms")
print(f"     proxy {alloc_proxy_GBps:.2f} GB/s is "
      f"{copy_bw_gbs/max(alloc_proxy_GBps,1e-9):.0f}x off -- as expected")

---
## Part 5: The KV cache, measured

The cell below grows the generation length and records peak memory at each
length, isolating the KV cache's **linear** growth. *(provided)*

> **Why the sweep goes to 512 tokens.** distilgpt2's KV cache is only 18,432
> bytes per token against a baseline of a couple of hundred MB. At 128 tokens the
> total growth is ~2 MB — close to run-to-run noise. By 512 tokens the reference
> T4 run grows **9.7 MB**, which is a slope you can actually fit. **This is a
> real measurement lesson:** if your signal is smaller than your noise, extend
> the sweep until it isn't, rather than quoting a number you cannot resolve.

You will then fit a least-squares line through `(new_tokens, peak_MB)` and compare
its slope against the analytic KV bytes-per-token. The grader notebook refits the
slope from your raw points, so the sweep you export has to be the sweep you ran.

In [ ]:
def peak_for_length(model, ids, n):
    """Peak allocated MB for generating n new tokens, measured from a clean slate."""
    if DEV == 'cuda':
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        model.generate(ids, max_new_tokens=n, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    if DEV == 'cuda':
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / 1e6
    return float('nan')

# Reach far enough that the KV cache clears the noise floor (see note above).
lengths = [16, 64, 128, 256, 512]
peaks = [peak_for_length(model, ids, n) for n in lengths]

peak_sweep = [[int(n), float(p)] for n, p in zip(lengths, peaks)]
pd.DataFrame({'new_tokens': lengths,
              'peak_MB': [round(p, 2) for p in peaks],
              'delta_vs_first_MB': [round(p - peaks[0], 2) for p in peaks]})

**💡 Hint: fitting the slope and the KV formula (Part 5.1)**

<details>
<summary>Click to reveal a hint</summary>

Least squares, by hand — no numpy needed, and the grader repeats exactly this:

```python
def fit_slope(xs, ys):
    n  = len(xs)
    mx = sum(xs) / n
    my = sum(ys) / n
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    den = sum((x - mx) ** 2 for x in xs)
    return num / den
```

For the analytic side, read the three numbers off `model.config` rather than
typing them in — `model.config.n_layer`, `model.config.n_head`, and
`model.config.n_embd // model.config.n_head`. Then:

```python
kv_bytes_per_token = 2 * N_LAYERS * N_HEADS * HEAD_DIM * BATCH * BYTES_PER
```

The leading 2 is K and V; the trailing one is bytes per FP16 element. For
distilgpt2 this must come out to exactly **18,432**.

</details>

In [ ]:
# 5.1: ★ TODO - fit a least-squares line through (new_tokens, peak_MB), then
#      compare its slope to the analytic KV bytes-per-token.
#
# Plain arithmetic rather than numpy.polyfit, so the grader can repeat it exactly
# with the standard library only.
def fit_slope(xs, ys):
    """Least-squares slope of ys vs xs (MB per token)."""
    # TODO 5.1a: return the least-squares slope
    #            sum((x-mx)(y-my)) / sum((x-mx)^2)
    raise NotImplementedError("Implement fit_slope")

slope_MB_per_token = fit_slope([p[0] for p in peak_sweep],
                               [p[1] for p in peak_sweep])

# TODO 5.1b: read the model's OWN config. Do not hard-code 6 / 12 / 64.
N_LAYERS  = None   # REPLACE: model.config.n_layer
N_HEADS   = None   # REPLACE: model.config.n_head
HEAD_DIM  = None   # REPLACE: model.config.n_embd // model.config.n_head
BATCH     = 1
# TODO 5.1b (cont.): read the element size off the model rather than hard-coding
#           2, so the formula follows the precision the model was actually
#           loaded in. Hint: next(model.parameters()).element_size()
BYTES_PER = None   # REPLACE: expect 2 at FP16

# TODO 5.1c: the analytic KV bytes per token.
#            2 (K and V) x layers x heads x head_dim x batch x bytes
kv_bytes_per_token = None  # REPLACE
kv_analytic_MB_per_token = None  # REPLACE: kv_bytes_per_token / 1e6

print(f"distilgpt2 config: {N_LAYERS} layers x {N_HEADS} heads x {HEAD_DIM} head_dim")
print(f"KV cache per token (analytic): {kv_bytes_per_token:,} bytes "
      f"= {kv_analytic_MB_per_token:.6f} MB")
print(f"  formula: 2 (K and V) x {N_LAYERS} x {N_HEADS} x {HEAD_DIM} x "
      f"{BATCH} x {BYTES_PER} bytes")
print()
print(f"Measured slope (fitted): {slope_MB_per_token:.6f} MB/token")
print(f"Ratio measured / analytic: "
      f"{slope_MB_per_token/kv_analytic_MB_per_token:.2f}x")
print()
# Segment-by-segment slopes: the KV cache is linear, but it is not the only
# thing growing. Watching the local slope CLIMB is how you see a second,
# super-linear term appear in your own data.
print("Local slope between adjacent sweep points:")
for (_a, _ya), (_b, _yb) in zip(peak_sweep, peak_sweep[1:]):
    _seg = (_yb - _ya) / (_b - _a)
    print(f"  {_a:4d} -> {_b:4d} tokens: {_seg:.6f} MB/token "
          f"({_seg/kv_analytic_MB_per_token:.2f}x analytic)")
print()
print("Two things to read off this:")
print("  1. The slope is POSITIVE and the growth is real -- that is the KV cache.")
print("  2. The slope is not constant: it climbs toward the long end. The KV")
print("     cache alone is exactly linear, so something super-linear is also")
print("     growing -- the O(seq^2) attention score matrix, which is a transient")
print("     intermediate rather than a cache.")
print()
print("So the fitted slope is the KV cache PLUS that term PLUS the allocator's")
print("block rounding, which is why it lands above the analytic figure. The")
print("grader notebook checks your slope against a refit of YOUR OWN sweep, never")
print("against the analytic value -- the gap is a real effect, not an error.")

In [ ]:
# --- Checkpoint 4 -------------------------------------------------------
# The KV formula is exact; the measured slope is graded loosely.
assert kv_bytes_per_token == 2 * N_LAYERS * N_HEADS * HEAD_DIM * BATCH * BYTES_PER
assert kv_bytes_per_token == 18432, (
    f"distilgpt2 should give 2*6*12*64*1*2 = 18,432 bytes/token, got "
    f"{kv_bytes_per_token:,}. Read the three config values from model.config "
    f"rather than typing them in.")

assert len(peak_sweep) >= 4, (
    f"the sweep needs at least 4 points to fit a slope, got {len(peak_sweep)}")

if cuda:
    # Peak memory must not DECREASE as you generate more tokens.
    _ys = [p[1] for p in peak_sweep]
    assert all(b >= a - 1.0 for a, b in zip(_ys, _ys[1:])), (
        f"peak memory went down as the generation got longer: {_ys}. "
        f"Did you reset_peak_memory_stats() before each length?")
    assert slope_MB_per_token > 0, (
        f"fitted slope is {slope_MB_per_token:.6f} MB/token. Peak memory must "
        f"grow with sequence length -- that growth IS the KV cache. If it came "
        f"out flat, your sweep may not reach far enough.")
    print(f"[OK] peak memory grows monotonically; slope "
          f"{slope_MB_per_token:.6f} MB/token > 0")
else:
    print("[OK] KV formula verified (CPU mode: no peak-memory sweep)")
print(f"     analytic KV: {kv_bytes_per_token:,} bytes/token")

---
## Part 5b: ★ TODO — KV-cache formula, attention variants, and scratchpad sizing

### Why the formula works

```
KV_bytes = 2 × n_layers × n_kv_heads × head_dim × seq_len × batch × bytes_per_element
           ^                ^
           K and V          MHA: = n_heads · GQA: < n_heads · MQA: = 1
```

Each layer caches one K and one V tensor (hence the leading 2), each of shape
`[batch, n_kv_heads, seq_len, head_dim]`. At FP16 that is 2 bytes per element.

### MHA vs GQA vs MQA

The baseline assumes **Multi-Head Attention**, where `n_kv_heads == n_heads`.
Modern architectures cut the number of *KV* heads while keeping all the query
heads, because the cache — not the query projection — is what grows with context:

| Attention Type | KV Heads | Example (32 layers, 32 q-heads, d=128, seq 2048, FP16) |
|---|---|---|
| **MHA** | `n_heads` (32) | 1073.7 MB |
| **GQA** (e.g. Llama-2 70B) | 8 | 268.4 MB — **4× smaller** |
| **MQA** (e.g. PaLM) | 1 | 33.6 MB — **32× smaller** |

> **Key insight:** GQA with 8 KV heads uses 4× less KV-cache memory than MHA with
> 32 heads, buying either 4× the context or 4× the batch in the same budget. The
> ratio is exactly `n_heads / n_kv_heads` — a pure consequence of the formula, so
> it is identical on every piece of hardware.

### The Spyre scratchpad constraint

A GPU keeps the KV cache in HBM (gigabytes) and can **page** it (PagedAttention).
Spyre has no HBM: an on-chip scratchpad of roughly **64 MB**, backed by DDR. The
weights want to be on-chip, the KV cache wants to be on-chip, and they do not
both fit. Deciding what occupies the scratchpad *at each point in time* is the
**placement problem** — a decision a GPU's hardware cache makes implicitly and
badly, and a dataflow compiler makes explicitly and must get right.

**💡 Hint: `kv_cache_bytes` for MHA / GQA / MQA (Part 5b.1)**

<details>
<summary>Click to reveal a hint</summary>

```python
def kv_cache_bytes(n_layers, n_kv_heads, d_head, seq_len, dtype_bytes=2, batch=1):
    return 2 * n_layers * n_kv_heads * d_head * seq_len * batch * dtype_bytes
```

The only parameter that changes between MHA, GQA and MQA is `n_kv_heads` — 32, 8
and 1 respectively for the Llama-2-style reference point. That is why the ratios
come out as exactly `32/8 = 4` and `32/1 = 32`, with no rounding.

</details>

In [ ]:
# 5b.1: ★ TODO - KV-cache size for MHA, GQA and MQA.
def kv_cache_bytes(n_layers, n_kv_heads, d_head, seq_len, dtype_bytes=2, batch=1):
    """KV-cache size in bytes for a single sequence (both K and V).

    Args:
        n_layers:    number of transformer layers
        n_kv_heads:  KV heads (= n_heads for MHA, < for GQA, = 1 for MQA)
        d_head:      dimension per head
        seq_len:     cached tokens
        dtype_bytes: 2 for FP16/BF16, 1 for INT8
        batch:       sequences cached concurrently
    """
    # TODO 5b.1a: implement the formula from the markdown above.
    raise NotImplementedError("Implement kv_cache_bytes")

# A Llama-2-style reference point, so the ratios are recognisable.
_L, _H, _D, _S = 32, 32, 128, 2048

# TODO 5b.1b: compute the three variants. Only n_kv_heads changes.
mha_bytes = None   # REPLACE: all 32 heads
gqa_bytes = None   # REPLACE: 8 KV heads
mqa_bytes = None   # REPLACE: 1 KV head

print(f"KV-cache at seq_len={_S}, {_L} layers, d_head={_D}, FP16:")
print(f"  MHA ({_H} KV heads): {mha_bytes/1e6:8.1f} MB")
print(f"  GQA ( 8 KV heads): {gqa_bytes/1e6:8.1f} MB  ({mha_bytes/gqa_bytes:.0f}x smaller)")
print(f"  MQA ( 1 KV head ): {mqa_bytes/1e6:8.1f} MB  ({mha_bytes/mqa_bytes:.0f}x smaller)")

# These ratios are exact arithmetic, identical on every machine.
assert mha_bytes == 2 * 32 * 32 * 128 * 2048 * 1 * 2
assert mha_bytes / gqa_bytes == 4.0
assert mha_bytes / mqa_bytes == 32.0
print("\n✓ KV-cache ratios are exactly n_heads / n_kv_heads (4x and 32x)")

In [ ]:
# 5b.2: ★ TODO - Spyre scratchpad sizing for the model you just profiled.
SPYRE_SCRATCHPAD_MB = 64     # on-chip SRAM budget, order of magnitude

# TODO 5b.2a: parameter count and the weight footprint at FP16.
n_params  = None   # REPLACE: sum(p.numel() for p in model.parameters())
weights_MB = None  # REPLACE: n_params * BYTES_PER / 1e6  (at the loaded dtype)

# TODO 5b.2b: if the KV cache got the WHOLE scratchpad, how much context fits?
#             Hint: an integer division of the budget by bytes-per-token.
max_seq_kv_only = None  # REPLACE

print(f"KV cache per token: {kv_bytes_per_token:,} bytes "
      f"({kv_bytes_per_token/1024:.1f} KB)")
print(f"Scratchpad budget:  {SPYRE_SCRATCHPAD_MB} MB")
print(f"Max context if KV gets ALL of it: {max_seq_kv_only:,} tokens")
print()
print(f"Model weights ({BYTES_PER} bytes/param): {weights_MB:.1f} MB "
      f"({n_params/1e6:.1f} M parameters)")

if weights_MB > SPYRE_SCRATCHPAD_MB:
    print(f"\n⚠  The weights ALONE ({weights_MB:.0f} MB) exceed the "
          f"{SPYRE_SCRATCHPAD_MB} MB scratchpad.")
    print("   -> The compiler cannot hold all weights on-chip at once.")
    print("   -> It must TILE them: stage a few layers in, compute, evict,")
    print("      stream the next layers from DDR.")
    print("   -> The KV cache competes with those weight tiles for the same SRAM.")
    print("   -> That is the PLACEMENT PROBLEM (Week 3): what lives where, when.")
else:
    _left = SPYRE_SCRATCHPAD_MB - weights_MB
    print(f"\n   Weights fit, leaving {_left:.1f} MB for the KV cache")
    print(f"   -> about {int(_left*1e6/kv_bytes_per_token):,} tokens of context")

print()
print("Per-token footprint grows LINEARLY, so the sequence length at which the")
print("KV cache stops being a rounding error is a division, not a guess. That is")
print("the arithmetic a dataflow compiler does before it schedules anything.")

In [ ]:
# --- Checkpoint 5 -------------------------------------------------------
# Everything here is fixed arithmetic: it must hold on any machine.
assert mha_bytes / gqa_bytes == 4.0, "MHA/GQA must be exactly 4x for 32 vs 8 KV heads"
assert mha_bytes / mqa_bytes == 32.0, "MHA/MQA must be exactly 32x for 32 vs 1 KV head"
assert kv_cache_bytes(6, 12, 64, 1) == 18432, "kv_cache_bytes is miscomputing distilgpt2"

# Doubling the sequence length must exactly double the cache (linearity).
assert kv_cache_bytes(6, 12, 64, 200) == 2 * kv_cache_bytes(6, 12, 64, 100), (
    "KV cache must be exactly linear in seq_len")

# Scratchpad arithmetic, recomputed.
assert max_seq_kv_only == int(SPYRE_SCRATCHPAD_MB * 1e6 / kv_bytes_per_token), (
    "max_seq_kv_only should be the scratchpad budget divided by bytes-per-token")
assert max_seq_kv_only > 0
print(f"[OK] KV arithmetic exact: 4x GQA, 32x MQA, linear in seq_len")
print(f"     {SPYRE_SCRATCHPAD_MB} MB scratchpad holds {max_seq_kv_only:,} tokens of KV")
print(f"     weights {weights_MB:.1f} MB vs scratchpad {SPYRE_SCRATCHPAD_MB} MB")

---

## Part 6: Annotate your memory bottlenecks

For your **top three** memory operators, record *why* each is memory-heavy and
*which* memory hierarchy you expect to serve it better. This replaces the
free-text table from earlier versions of the lab: you pick from a fixed
vocabulary, so the answer can be checked automatically and you get your
self-assessment feedback immediately.

**Pick each `op` from your own Part 3 table** — the grader checks that the
operators you annotate are ones your run actually profiled.

| `why` code | Means |
|---|---|
| `low_arithmetic_intensity` | Few FLOPs per byte moved; the op is bandwidth-starved, not compute-starved |
| `large_intermediate_tensor` | Materializes a big temporary (e.g. the O(seq²) attention score matrix) |
| `weight_reread_per_token` | Re-reads the same weight matrix for every decoded token (batch=1 decode) |
| `random_access_low_reuse` | Scattered, data-dependent addresses with little locality (e.g. embedding gather) |
| `kv_cache_growth` | Its footprint grows with sequence length because it carries the KV cache |

| `better_on` code | Means |
|---|---|
| `spyre` | A compiler-placed scratchpad wins: the access pattern is regular and tileable |
| `gpu` | A hardware-managed cache wins: the pattern is irregular or data-dependent |
| `comparable` | No clear winner; both hierarchies move about the same bytes |

> **There is no single right assignment**, and the grader notebook does not check
> *which* codes you choose — only that your three operators come from your own
> profile, are distinct, and that every code is from the lists above. The
> reasoning behind the choice is what the **Concept Check quiz** tests, and that
> quiz *is* the graded one (10 graded points, 80% to pass).

In [ ]:
# 6.1: ★ TODO - annotate your top three memory operators.
# The vocabularies are fixed so this can be graded without reading prose.
BOTTLENECK_WHY = {
    "low_arithmetic_intensity",
    "large_intermediate_tensor",
    "weight_reread_per_token",
    "random_access_low_reuse",
    "kv_cache_growth",
}
BOTTLENECK_TARGET = {"spyre", "gpu", "comparable"}

_ops = list(top['op'])
print("Your top memory operators (pick three of these):")
for _i, _o in enumerate(_ops):
    print(f"  [{_i}] {_o}")

# TODO 6.1: three entries, each {"op": ..., "why": ..., "better_on": ...}.
#           - every "op" must come from YOUR _ops list above (use _ops[i])
#           - "why" from BOTTLENECK_WHY, "better_on" from BOTTLENECK_TARGET
#           - the three operators must be distinct
#           There is no single right assignment. Choose what your own profile
#           and the Week 4 lecture support; the grader checks form, not opinion.
bottlenecks = [
    # {"op": _ops[0], "why": "...", "better_on": "..."},
]

for _b in bottlenecks:
    print(f"  {_b['op']:24s} {_b['why']:26s} -> {_b['better_on']}")

In [ ]:
# --- Checkpoint 6 -------------------------------------------------------
assert isinstance(bottlenecks, list) and len(bottlenecks) == 3, (
    f"bottlenecks must be a list of exactly 3 entries, got {len(bottlenecks)}")

_seen = set()
for _i, _b in enumerate(bottlenecks):
    assert set(_b) == {"op", "why", "better_on"}, (
        f"entry {_i} should have exactly the keys op, why, better_on; got {sorted(_b)}")
    assert _b["op"] in _ops, (
        f"entry {_i}: '{_b['op']}' is not in your Part 3 table. Annotate "
        f"operators your own run profiled: {_ops[:5]}")
    assert _b["why"] in BOTTLENECK_WHY, (
        f"entry {_i}: why='{_b['why']}' is not one of {sorted(BOTTLENECK_WHY)}")
    assert _b["better_on"] in BOTTLENECK_TARGET, (
        f"entry {_i}: better_on='{_b['better_on']}' is not one of "
        f"{sorted(BOTTLENECK_TARGET)}")
    assert _b["op"] not in _seen, f"'{_b['op']}' is annotated twice; pick three distinct ops"
    _seen.add(_b["op"])

print("[OK] 3 distinct operators annotated, all from your own profile, "
      "all codes valid")

---
## Part 7: ★ TODO — Update the Accelerator Landscape Map

Add this week's row to the shared map you have been building since Lab 1. Week 4's
contribution is the **memory hierarchy** column.

In [ ]:
# 7.1: ★ TODO - Landscape Map row: the memory hierarchy you profiled this week.
landscape_row_w4 = {
    "accelerator":        None,  # e.g. (f"GPU ({torch.cuda.get_device_name(0)})" if cuda else "CPU")
    "memory_hierarchy":   None,  # one short phrase, e.g. "L1/L2 + GDDR/HBM (hardware-managed cache)"
    "peak_alloc_MB":      None,  # round(peak_bytes / 1e6, 1)
    "kv_bytes_per_token": None,  # int(kv_bytes_per_token) from Part 5
    "measured_bw_GBps":   None,  # round(copy_bw_gbs, 1)  <- the 4.2 measurement, NOT the 4.1 proxy
    "top_mem_operator":   None,  # str(top.iloc[0]['op'])
}
for _k, _v in landscape_row_w4.items():
    print(f"  {_k:20s}: {_v}")

In [ ]:
# --- Checkpoint 7 -------------------------------------------------------
_required = {"accelerator", "memory_hierarchy", "peak_alloc_MB",
             "kv_bytes_per_token", "measured_bw_GBps", "top_mem_operator"}
assert set(landscape_row_w4) == _required, (
    f"landscape_row_w4 keys are wrong: {sorted(set(landscape_row_w4) ^ _required)}")
assert all(v is not None for v in landscape_row_w4.values()), (
    "every landscape_row_w4 field must be filled in")

# The measured fields must mirror what you actually measured. The graded
# notebook re-checks each of these against the raw values.
assert abs(landscape_row_w4["peak_alloc_MB"] - peak_bytes / 1e6) < 0.5, (
    "peak_alloc_MB should be peak_bytes / 1e6, rounded to 1 decimal")
assert landscape_row_w4["kv_bytes_per_token"] == kv_bytes_per_token
assert abs(landscape_row_w4["measured_bw_GBps"] - copy_bw_gbs) < 0.5, (
    "measured_bw_GBps should be your Part 4.2 bandwidth, not the 4.1 proxy")
assert landscape_row_w4["top_mem_operator"] == str(top.iloc[0]['op'])
print("[OK] Week 4 Landscape Map row complete and consistent with your measurements")

---

## Check your understanding (ungraded)

These prompts are **not graded and not submitted** — nothing here travels in your
JSON, and no one reviews your answers. Work through them anyway: they are the
reasoning the **Concept Check quiz** tests, and that quiz *is* the graded one,
worth **10 graded points**. Answer in your own words, citing the numbers *you*
measured, then compare against the solution notebook once it unlocks.

---

### 1. Why is the allocator-activity proxy not bandwidth?

You computed two numbers in Part 4: `|self_mem_MB| / time` for your top operators,
and a measured bandwidth from a copy whose byte count you knew analytically. They
differ by roughly an order of magnitude. Explain *why* — what does
`self_device_memory_usage` actually count, and which of {reads, writes, net
allocation, in-place updates} does it miss?

*Your reasoning here (ungraded — for your own learning).*

---

### 2. Does your measured per-token growth match the formula?

You fitted a slope to `peak_MB` vs `new_tokens` and compared it to
`2 × layers × heads × head_dim × 2 bytes`. If your measured slope came out
*above* the analytic KV number, where did the extra bytes come from? If it came
out *below*, what would that imply about when the allocator actually releases
memory?

*Your reasoning here (ungraded — for your own learning).*

---

### 3. The scratchpad question (the Landscape Map question).

distilgpt2's weights alone exceed Spyre's ~64 MB on-chip scratchpad, and the KV
cache has to share whatever is left. Name the decision this forces on the
compiler that a GPU never has to make, and say what it costs when it gets it
wrong. Then predict: at what sequence length does the KV cache stop being a
rounding error and start driving the placement decision?

You will revisit this in Week 6. It's okay to be wrong, but be *concrete*.

*Your reasoning here (ungraded — for your own learning).*

---

## Part 8: Save and export your results

The cell below writes **`lab4_results.json`** and prints it. Copy its contents
into the `RESULTS_JSON` cell of the **grader notebook** on Coursera Labs and run
every cell there to see your **40 ungraded Self-Assessment points**.

It carries the **raw inputs** next to every derived number — the byte counts, the
sweep points, your model config — so the grader can recompute each derived value
rather than take it on trust.

> Those 40 points are **feedback, not a grade** — they tell you how well the lab
> went. The graded item for this week is the **Concept Check quiz** (10 points,
> auto-graded on Coursera, 80% to pass).

In [ ]:
# === Part 8: Save your results for submission ==========================
# This writes lab4_results.json. You will paste its contents into the grader
# notebook on Coursera Labs. This file IS your submission: without it there is
# nothing to grade.
#
# PROVIDED - do not edit. Everything here is built from variables you already
# computed in Parts 3-7, so there is nothing to fill in. Just run it.
import json as _json

RESULTS = {
    # --- model identity: lets the grader recompute the KV formula from YOUR
    #     config instead of assuming distilgpt2 ---------------------------
    "model_config": {
        "model_name": MODEL_NAME,
        "n_layer":    int(N_LAYERS),
        "n_head":     int(N_HEADS),
        "n_embd":     int(model.config.n_embd),
        "head_dim":   int(HEAD_DIM),
        "n_params":   int(n_params),
        "bytes_per_element": int(BYTES_PER),
    },

    # --- Part 3: peak allocation and the memory-heavy operators ----------
    # peak_bytes is raw bytes; peak_alloc_MB is the rounded display value and
    # the grader checks they round-trip.
    "peak_bytes":   int(peak_bytes),
    "peak_alloc_MB": round(peak_bytes / 1e6, 1),
    "top_ops": [[str(r['op']), float(r['self_mem_MB']), float(r['time_ms']),
                 int(r['calls'])] for _, r in top.iterrows()],

    # --- Part 4: the wrong number and the right one ----------------------
    "alloc_proxy_GBps": float(alloc_proxy_GBps),
    "copy_bw": {
        "size_mb":     float(COPY_SIZE_MB),
        "bytes_moved": int(copy_bytes_moved),     # = 2 x size_mb (read + write)
        "median_ms":   float(copy_time_ms),
        "bw_gbs":      float(copy_bw_gbs),        # recomputed by the grader
    },

    # --- Part 5: the sweep, raw. The grader REFITS the slope from these
    #     points, so they must be the points you actually measured. -------
    "peak_sweep": peak_sweep,
    "kv": {
        "bytes_per_token":       int(kv_bytes_per_token),
        "analytic_MB_per_token": float(kv_analytic_MB_per_token),
        "slope_MB_per_token":    float(slope_MB_per_token),
    },

    # --- Part 5b: exact arithmetic, hardware-independent -----------------
    "kv_variants": {
        "ref_layers": int(_L), "ref_heads": int(_H),
        "ref_head_dim": int(_D), "ref_seq_len": int(_S),
        "mha_bytes": int(mha_bytes),
        "gqa_bytes": int(gqa_bytes),
        "mqa_bytes": int(mqa_bytes),
        "mha_over_gqa": float(mha_bytes / gqa_bytes),
        "mha_over_mqa": float(mha_bytes / mqa_bytes),
    },
    "scratchpad": {
        "total_MB":        float(SPYRE_SCRATCHPAD_MB),
        "bytes_per_token": int(kv_bytes_per_token),
        "max_seq_kv_only": int(max_seq_kv_only),
        "weights_MB":      float(weights_MB),
    },

    # --- Part 6: controlled-vocabulary bottleneck annotation -------------
    "bottlenecks": bottlenecks,

    # --- Part 7: your Landscape Map row ---------------------------------
    "landscape_row_w4": landscape_row_w4,

    # --- environment: reported, never graded against --------------------
    "environment": {
        "device": (torch.cuda.get_device_name(0) if cuda else "CPU only"),
        "torch_version": torch.__version__,
        "mode": MODE,
    },
}

with open("lab4_results.json", "w") as _f:
    _json.dump(RESULTS, _f, indent=2)

print("Saved lab4_results.json\n")
print(_json.dumps(RESULTS, indent=2))
print("\n" + "=" * 70)
print("NEXT STEPS")
print("=" * 70)
print("1. Copy the JSON printed above (from the opening '{' through the")
print("   final '}'), or open the file browser (folder icon, left sidebar)")
print("   and double-click lab4_results.json.")
print("2. Open the grader notebook on Coursera Labs.")
print("3. Paste the contents into the RESULTS_JSON cell, then run every cell.")
print("   It reports your 40 ungraded Self-Assessment points.")
print("4. The Concept Check quiz is a SEPARATE auto-graded Coursera item")
print("   worth 10 points. Take it as well.")
print("\nKeep this Colab notebook: if a graded check fails you will want to")
print("re-run the cell that produced the value.")

# Optional: download the file instead of copy-pasting.
# from google.colab import files
# files.download('lab4_results.json')

---
## ★ Optional: Dataflow Deep Dive (Spyre hardware)
_Requires Spyre access via an IBM Research partnership. Skip on the core track — it earns no extra credit._

1. Run the same generation on Spyre.
2. Use **hardware performance counters** to capture real memory traffic between scratchpad and DDR.
3. Compare the measured traffic to your GPU-profiler estimates from Parts 3–4.
4. Identify where **explicit scratchpad placement** measurably changes the memory pattern versus the GPU — and at least one operator where it does **not** — and explain each in terms of the hierarchy.

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.

In [ ]:
# Spyre track only, pseudocode skeleton; consult your partnered-program setup guide.
# import torch_sendnn          # registers the 'aiu'/'spyre' device
# spyre = 'aiu'
# model_s = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(spyre).eval()
# ids_s = tok(PROMPT, return_tensors='pt').input_ids.to(spyre)
# counters = capture_hw_counters(lambda: model_s.generate(ids_s, max_new_tokens=48))
# compare(counters, top)   # scratchpad<->DDR traffic vs GPU estimate

---

### Wrap-up checklist
- [ ] **Runtime → Change runtime type → T4 GPU** set before running anything.
- [ ] Runtime → Run all; every **Checkpoint** prints `[OK]`.
- [ ] Parts 2–4 implemented; operator table and measured bandwidth present.
- [ ] Part 5 sweep run and slope fitted; Part 5b arithmetic passes.
- [ ] Part 6 annotation filled from your own operators; Part 7 row complete.
- [ ] Part 8 run, `lab4_results.json` written and copied.
- [ ] JSON pasted into the grader notebook on Coursera, which reports your 40 ungraded Self-Assessment points.
- [ ] **Concept Check quiz** taken — a separate auto-graded Coursera Graded
      Assignment worth **10 graded points**. Mandatory, 80% to pass.

_End of Lab 4. Next week: debugging failures across the dataflow stack._

---

*Course: Dataflow Architectures for AI, Week 4 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*